# Parameter Estimation Tutorial

## Overview

This notebook demonstrates how to perform parameter estimation for precipitation 
conversion models using the data we obtained from Minteq 
(equilibrium simulation of precipitation). 

The parameter estimation workflow in this tutorial includes:

- Defining a mathematical model for precipitation conversion
- Importing all the necessary libraries and data
- Solving the estimation problem using least-squares methods for one species
- Visualizing model predictions
- Extending parameter estimation to multiple species
- Comparing results with the experimental data

## Mathematical Model

The precipitation conversion model is defined as:

$Conversion = \exp\left(-\frac{\epsilon}{\text{Oxalic Acid Dosage}^{n_{DA}}}\right)$

where:

- Conversion: the fraction of species precipitated
- Oxalic Acid Dosage: the amount of oxalic acid added into the precipitator
- $\epsilon$: a fitted model parameter
- $n_{DA}$: a fitted model parameter

The objective of this tutorial is to determine the values of $\epsilon$ and $n_{DA}$ that
represent the experimental data well.

## Data Source

The data used in this tutorial were generated using equilibrium simulations
from Minteq. These data represent precipitation behavior of critical minerals under different
oxalic acid dosages.

## Learning Objectives

After completing this tutorial, users should be able to:

1. Import the necessary library
2. Estimate parameters for one species
3. Extend parameter estimation to multiple species
4. Visualize results and compare with experimental data

## Step 1: Import Libraries

In this section, we import the Python packages required for parameter estimation,
data handling, and visualization. The tutorial uses the following major libraries:

 - Pyomo
 - IDAES
 - NumPy
 - Pandas
 - Matplotlib

In [ ]:
# Import Pyomo's math and parameter estimation libraries
import pyomo.environ as pyo
from pyomo.environ import exp, log
import pyomo.contrib.parmest.parmest as parmest
from pyomo.contrib.parmest.experiment import Experiment

# Import degrees of freedom function from IDAES for sanity check
from idaes.core.util.model_statistics import degrees_of_freedom

<div class="alert alert-block alert-info">
<b>Inline Exercise:</b>
import `pandas` as pd, `numpy` as nd and `matplotlib.pyplot` as plt. 
</div>

In [ ]:
# TODO: import the libraries mentioned above

In [ ]:
# TODO: import the libraries mentioned above
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

## Step 2: Estimate Parameters

This section demonstrates the workflow to estimate model parameters for one species using the Pyomo Parmest framework.

The workflow consists of:

- Import data
- Build the parameter estimation problem
- Solve for optimal parameters
- Evaluate results

The objective is to determine model parameters that minimize the
difference between experimental data and prediction results.

### 2.1 Read Data File and Display

We import precipitation conversion data from a CSV file.

The dataset contains:

- Oxalic acid dosage amount
- Precipitation conversion for each species

<div class="alert alert-block alert-info">
<b>Inline Exercise:</b>
import the data csv file as data, and display the data 
</div>

In [ ]:
# Load data from csv
data = pd.read_csv("data_precipitation.csv")

# TODO: Display the dataset

In [ ]:
# Load data from csv
data = pd.read_csv("data_precipitation.csv")

# TODO: Display the dataset
display(data)

### 2.2 Define the Model for Precipitation

We define the mathematical model used for precipitation simulation.

The model describes precipitation conversion as a function of oxalic
acid dosage using a nonlinear expression:

$Conversion = \exp\left(-\frac{\epsilon}{\text{Oxalic Acid Dosage}^{n_{DA}}}\right)$

The unknown parameters are:

- $\epsilon$ (eps) — a model coefficient controlling precipitation intensity
- n_DA — an exponent describing the acid dosage sensitivity


In [ ]:
def isotherm():

    # Todo: Create a ConcreteModel object
    m = pyo.ConcreteModel()

    # Set up variables of the model
    m.n_DA = pyo.Var(initialize=2)
    m.eps = pyo.Var(initialize=1)
    # Conversion variable
    m.CM = pyo.Var(initialize=0.1, bounds=(1e-8, 1))
    m.oxalic = pyo.Var(initialize=0.1, bounds=(1e-8, 30))
   
    # Set bounds on variables
    m.n_DA.setlb(1)
    m.n_DA.setub(5)

    m.eps.setlb(1)
    m.eps.setub(5)
    
    m.CM_rule = pyo.Constraint(
        expr=(m.CM == exp(-(m.eps/(m.oxalic))**m.n_DA)))

    # Fix variables to have 0 Degrees of Freedom (DoF)
    m.n_DA.fix(2)
    m.eps.fix(1)
    m.oxalic.fix(1.425119)
    
    # Return initialized flash model
    return m

### 2.3 Define Nd Precipitation Experiment 

We define a class for the precipitation system.

Each experiment corresponds to one row of the dataset and contains:

- Input (oxalic acid dosage)
- Measured outputs (conversion)

In [ ]:
# Create precipitation class
class PrecipitationExperiment(Experiment):

    # Define data 
    def __init__(self, data, experiment_number):
        self.data = data
        self.experiment_number = experiment_number
        self.data_i = data.iloc[experiment_number, :]
        self.model = None

    # Create a model based on the isotherm model
    def create_model(self):
        self.model = m = isotherm()
        return m

    # Define inputs and outputs of the experiment
    def finalize_model(self):
        m = self.model

        # Experiment inputs values
        m.oxalic = self.data_i['Oxalate-2'] 

        # Experiment output values
        m.CM = self.data_i['Nd2(C2O4)3(s)']

        return m

    # Label the parameters to be calculated
    def label_model(self):
        m = self.model

        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_outputs.update(
            [
                (m.CM, self.data_i['Nd2(C2O4)3(s)']),
            ]
        )

        # Specify unknown parameters
        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.unknown_parameters.update(
            (k, pyo.ComponentUID(k)) for k in [m.n_DA, m.eps]
        )

        return m

    def get_labeled_model(self):
        m = self.create_model()
        m = self.finalize_model()
        m = self.label_model()

        return m

### Sanity Check

After we define the precipitator unit model, we test the model and do sanity check to ensure the degrees of freedom in the system is zero.

In [ ]:
# Testing the initialized model
test_data = {"oxalic": 1.425119}

m = isotherm()

m.oxalic = test_data["oxalic"]

# Check that degrees of freedom is 0
print("Degrees of Freedom = ", degrees_of_freedom(m))

### 2.4 Finish setting up the experiment

We complete the parameter estimation problem definition here.

This includes:

- Defining the parameters to be estimated
- Defining the objective function

The objective function is defined as the sum of squared errors (SSE)
between:

- Experimental conversion values
- Model predicted conversion values


<div class="alert alert-block alert-info">
<b>Inline Exercise:</b>
Create a list called `variable_name` with the $n_{DA}$, $\epsilon$ above-mentioned variables declared as strings.
</div>

In [ ]:
# Todo: Create a list of vars to estimate

In [ ]:
# Todo: Create a list of vars to estimate
variable_name = [
    "n_DA",
    "eps",
]

We need to provide a method to return an expression to compute the sum of squared errors that will be used as the objective in solving the parameter estimation problem. For this problem, the error will be computed for the conversion of REEs from ions to oxides.

<div class="alert alert-block alert-info">
<b>Inline Exercise:</b>
Complete the following cell by adding an expression to compute the sum of square errors. 
</div>

In [ ]:
# Create method to return an expression that computes the sum of squared error
# def SSE(m, data):
    # Todo: Add expression for computing the sum of squared errors.
    # In this case y is the experimental data, while y_hat is the model solution.
    # The SSE equation should be the sum of all SSE of m.experiment_outputs.items()

In [ ]:
# Create method to return an expression that computes the sum of squared error
def SSE(m):
    expr = sum(((y - y_hat))**2 for y, y_hat in m.experiment_outputs.items())
    return expr * 1e4

### 2.5 Solve the Parameter Estimation Model

We solve the parameter estimation problem.

The steps include:

1. Create a list of experiments
2. Initialize a Parmest Estimator object
3. Solve the optimization problem

Parmest function automatically:

- Builds the optimization model
- Aggregates all experiments
- Minimizes the objective function

The solver determines the parameter values that best fit the
imported data.

In [ ]:
# Initialize a parameter estimation object
exp_list = []
for i in range(data.shape[0]):
    exp_list.append(PrecipitationExperiment(data, i))

# Call solver
solver_options = {"tol": 1e-8}    

# Define Estimator
pest = parmest.Estimator(exp_list, obj_function=SSE, solver_options=solver_options, tee=True)

# Run parameter estimation using all data
obj_value, parameters = pest.theta_est()

### 2.6 Display Results

After solving the parameter estimation problem, the optimized parameter values
are displayed here.

The estimated parameters will be used in the next section to evaluate
model performance and generate plots.

In [ ]:
print()
print("The values for the parameters are as follows:")
for k, v in parameters.items():
    print(k, "=", v)

## Step 3: Visualizing Results

After estimating the model parameters, it is important to evaluate how
well the model fits the actual data.

This section generates plots that compare:

- Measured conversion values
- Model predicted conversion values

The following types of plots are generated:

- Conversion vs. acid dosage amount plots
- Parity plots

These plots allow users to determine whether the estimated parameters
provide an adequate representation of the precipitation behavior.

In [ ]:
# Give calculated values to the variables
n_DA = {"Nd2(C2O4)3(s)":2.419137}
eps = {"Nd2(C2O4)3(s)":1.01030}

# Create an array for Oxalic acid dosage
oxalate = np.linspace(0.05, 30, 50)

# Create an empty list
Nd = []

#Caluclate conversion for Nd oxalate
for i in range(len(oxalate)):
    Ndi = exp(-(eps["Nd2(C2O4)3(s)"]/(oxalate[i]))**n_DA["Nd2(C2O4)3(s)"]) 
    
    #Store results
    Nd.append(Ndi)

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)

ax.plot(data["Oxalate-2"],data["Nd2(C2O4)3(s)"],label="Nd2(C2O4)3(s)")
ax.scatter(oxalate,Nd)

plt.xlabel('Acid Dosage')
plt.ylabel('Conversion')
plt.title('Conversion of Nd to ND oxalate in precipitator')

ax.legend(title='CM')
plt.show

## Parity Plot

After the conversion under a function of acid dosage plot is generated, the parity plot is created to compare the actual values and prediction values.

The parity plot shows:

- Experimental conversion values on the horizontal axis
- Model predicted conversion values on the vertical axis

A diagonal line represents perfect agreement between:

- Model predictions
- Experimental data

Points close to the diagonal indicate accurate predictions, while
points far from the line indicate discrepancies.

Parity plots provide a compact and effective method for evaluating the
quality of parameter estimation results.

In [ ]:
# Defining array with experimental acid dosage
oxalic_acid_pp = data["Oxalate-2"].to_numpy()

# Creating empty list to store results
Nd_pp = []

#Re run equation to calculate conversion at experimental acid dosage
for i in range(len(oxalic_acid_pp)):
    Ndi_pp = exp(-(eps["Nd2(C2O4)3(s)"]/(oxalic_acid_pp[i]))**n_DA["Nd2(C2O4)3(s)"]) 

    # Store results
    Nd_pp.append(Ndi_pp)

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)

ax.scatter(Nd_pp,data["Nd2(C2O4)3(s)"],label="Nd2(C2O4)3(s)")
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Identity Line')

plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Parity Plot')

ax.legend(title='CM')
plt.show

## Step 5: Estimating Parameters for Multiple Species

This section extends the parameter estimation model from one species to multiple
species.

This allows:

- Consistent parameter estimation for each species
- Better representation of the real-world systems

The workflow follows the similar steps as the single-species case but with
a larger model.

## 5.1 Create a New Model with Multiple Species 

We define a new Pyomo model that includes multiple species.

Each species has its own set of parameters:

- eps[i]
- n_DA[i]

The model calculates conversion for each species as a
function of oxalic acid dosage.

This formulation allows all species to be estimated simultaneously
within a single optimization model.

In [ ]:
def isotherm_multiple():

    # Todo: Create a ConcreteModel object
    m = pyo.ConcreteModel()

    # Species names in the order they map to the integer index used below
    m.species_names = [
        "La2(C2O4)3(s)",
        "Nd2(C2O4)3(s)",
        "Gd2(C2O4)3(s)",
        "Ce2(C2O4)3(s)",
        "Y2(C2O4)3(s)",
        "Sm2(C2O4)3(s)",
        "Sc2(C2O4)3(s)",
        "Pr2(C2O4)3(s)",
        "Dy2(C2O4)3(s)",
    ]

    # parmest requires the first index of experiment_outputs to be an int/float
    # data point, so the species are indexed by integer position instead of name
    m.oxalates = list(range(len(m.species_names)))

    # Dictionaries with initial values to our parameters
    # parameter based on pH 1.5
    eps_init = {
            "Sc2(C2O4)3(s)": 6.4,
            "Y2(C2O4)3(s)": 4.5,
            "La2(C2O4)3(s)": 4.3,
            "Ce2(C2O4)3(s)": 1.1,
            "Pr2(C2O4)3(s)": 2.09,
            "Nd2(C2O4)3(s)": 1,
            "Sm2(C2O4)3(s)": 2.3,
            "Gd2(C2O4)3(s)": 3.1,
            "Dy2(C2O4)3(s)": 4.9,
    }
        
    # parameter based on pH 1.5
    N_D_init = {
            "Sc2(C2O4)3(s)": 6,
            "Y2(C2O4)3(s)": 4,
            "La2(C2O4)3(s)": 4,
            "Ce2(C2O4)3(s)": 2,
            "Pr2(C2O4)3(s)": 3,
            "Nd2(C2O4)3(s)": 2,
            "Sm2(C2O4)3(s)": 3,
            "Gd2(C2O4)3(s)": 4,
            "Dy2(C2O4)3(s)": 4,
        }
        
    # Create variables for parmeters and variables, note they are indexed by integer position
    m.n_DA = pyo.Var(
        m.oxalates,
        initialize={i: eps_init[name] for i, name in enumerate(m.species_names)},
        bounds=(1, 7),
    )
    m.eps = pyo.Var(
        m.oxalates,
        initialize={i: N_D_init[name] for i, name in enumerate(m.species_names)},
        bounds=(1, 7),
    )
    m.CM = pyo.Var(m.oxalates, initialize=0.1, bounds=(1e-8, 1))
    m.oxalic = pyo.Var(initialize=0.1, bounds=(1e-8, 30))

    # Create constraint now indexed by our list
    @m.Constraint(
    m.oxalates,
    doc="Conversion constraint",
    )
    def conversion_constraint(blk, t):
        return (m.CM[t] == exp(-(m.eps[t]/(m.oxalic))**m.n_DA[t]))
    
    # Fix variables to set up square problem
    fixed_n_DA = {
        "La2(C2O4)3(s)": 4.9,
        "Nd2(C2O4)3(s)": 2,
        "Gd2(C2O4)3(s)": 4.2,
        "Ce2(C2O4)3(s)": 2.7,
        "Y2(C2O4)3(s)": 4.7,
        "Sm2(C2O4)3(s)": 3.7,
        "Sc2(C2O4)3(s)": 2.3,
        "Pr2(C2O4)3(s)": 3.4,
        "Dy2(C2O4)3(s)": 4.7,
    }
    fixed_eps = {
        "La2(C2O4)3(s)": 4.3,   
        "Nd2(C2O4)3(s)": 1,
        "Gd2(C2O4)3(s)": 3.1,    
        "Ce2(C2O4)3(s)": 1.1,
        "Y2(C2O4)3(s)": 4.5,    
        "Sm2(C2O4)3(s)": 2.3,
        "Sc2(C2O4)3(s)": 6.4,
        "Pr2(C2O4)3(s)": 2.1,
        "Dy2(C2O4)3(s)": 4.9
        }
    for i, name in enumerate(m.species_names):
            m.eps[i].fix(fixed_eps[name])
            m.n_DA[i].fix(fixed_n_DA[name])

    m.oxalic.fix(1.425119)

    # Return initialized flash model
    return m

## 5.2 Define the Experiments with Multiple Species

We define a new Experiment class for the multi-species
model.

Each experiment now contains:

- An input condition (oxalic acid dosage amount)
- Multiple model outputs (species conversions)


In [ ]:
class PrecipitationMultipleExperiment(Experiment):

    def __init__(self, data, experiment_number):
        self.data = data
        self.experiment_number = experiment_number
        self.data_i = data.iloc[experiment_number, :]
        self.model = None

    def create_model(self):
        self.model = m = isotherm_multiple()
        return m

    def finalize_model(self):
        m = self.model

        # Experiment inputs values
        m.oxalic = self.data_i['Oxalate-2'] 

        # Experiment output values, mapped by integer index to species name
        for i, name in enumerate(m.species_names):
            m.CM[i] = self.data_i[name]

        return m

    def label_model(self):
        m = self.model

        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_outputs.update(
            [
                (m.CM[i], self.data_i[name])
                for i, name in enumerate(m.species_names)
            ]
        )

        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.unknown_parameters.update(
            (k, pyo.ComponentUID(k))
            for i in m.oxalates
            for k in [m.n_DA[i], m.eps[i]]
        )

        return m

    def get_labeled_model(self):
        m = self.create_model()
        m = self.finalize_model()
        m = self.label_model()

        return m

## Sanity Check

Again, we do the sanity check test to ensure the degree of freedom in the system is 0.

In [ ]:
from idaes.core.util.model_statistics import degrees_of_freedom

# Testing the initialized model
test_data = {"oxalic": 1.425119}

m1 = isotherm_multiple()

m1.oxalic = test_data["oxalic"]

<div class="alert alert-block alert-info">
<b>Inline Exercise:</b>
Let's check that the degrees of freedom are = o. 
</div>

In [ ]:
# TODO: Check that degrees of freedom is 0

In [ ]:
# TODO: Check that degrees of freedom is 0
print(degrees_of_freedom(m1))

Define the new large array of variables for each species.

In [ ]:
variable_name = [
    "n_DA['La2(C2O4)3(s)']",
    "n_DA['Nd2(C2O4)3(s)']",
    "n_DA['Gd2(C2O4)3(s)']",
    "n_DA['Ce2(C2O4)3(s)']",
    "n_DA['Y2(C2O4)3(s)']",
    "n_DA['Sm2(C2O4)3(s)']",
    "n_DA['Sc2(C2O4)3(s)']",
    "n_DA['Pr2(C2O4)3(s)']",
    "n_DA['Dy2(C2O4)3(s)']",
    "eps['La2-(C2O4)3(s)']",
    "eps['Nd2-(C2O4)3(s)']",
    "eps['Gd2(C2O4)3(s)']",
    "eps['Ce2(C2O4)3(s)']",
    "eps['Y2(C2O4)3(s)']",
    "eps['Sm2(C2O4)3(s)']",
    "eps['Sc2(C2O4)3(s)']",
    "eps['Pr2(C2O4)3(s)']",
    "eps['Dy2(C2O4)3(s)']"
]

## 5.3 Solve the Parameter Estimation Model

We solve the multi-species parameter estimation problem.

The procedure is similar to the single-species case.

The solver would determine parameter values that best match the experimental
conversion data for all species simultaneously.

In [ ]:
# Initialize a parameter estimation object
exp_list1 = []
for i in range(data.shape[0]):
    exp_list1.append(PrecipitationMultipleExperiment(data, i))

solver_options = {"tol": 1e-8}    

<div class="alert alert-block alert-info">
<b>Inline Exercise:</b>
As with the example above let's write and run the estimator function. 
</div>

In [ ]:
# TODO: Create an estimator object 


# TODO: Run parameter estimation using all data

In [ ]:
# TODO: Create an estimator object 
pest = parmest.Estimator(exp_list1, obj_function=SSE, solver_options=solver_options, tee=True)

# TODO: Run parameter estimation using all data
obj_value, parameters = pest.theta_est()

## Result Display

The optimized parameters for all the species are displayed.

In [ ]:
print()
print("The values for the parameters are as follows:")
species_names = exp_list1[0].model.species_names
for k, v in parameters.items():
    label = str(k)
    for i, name in enumerate(species_names):
        label = label.replace(f"[{i}]", f"[{name}]")
    print(f"{label} = {v:.4f}")

## 5.4 Calculate Precipitation Isotherm with Optimized Parameters

After solving the parameter estimate model, the precipitation isotherm is developed over a
range of oxalic acid dosage values.

The calculated results show how conversion varies
with acid dosage for each species.

In [ ]:
oxalate = np.linspace(0.05, 30, 50)

La = []
Nd = []
Gd = []
Ce = []
Y = []
Sm = []
Sc = []
Pr = []
Dy = []
Ca = []
Fe = []

n_DA = {"La2(C2O4)3(s)":4.6340,
        "Nd2(C2O4)3(s)":2.4191,
        "Gd2(C2O4)3(s)":4.1995,
        "Ce2(C2O4)3(s)":2.737238,
        "Y2(C2O4)3(s)":4.67403,
        "Sm2(C2O4)3(s)":3.7201,
        "Sc2(C2O4)3(s)":2.3606,
        "Pr2(C2O4)3(s)":3.44364,
        "Dy2(C2O4)3(s)":4.73106,
        "Ca(C2O4)(s)":4.45302,
        "FE2(C2O4)3(s)":3.6495}

eps = {"La2(C2O4)3(s)":4.3717,
        "Nd2(C2O4)3(s)":1.01030,
        "Gd2(C2O4)3(s)":3.07276,
        "Ce2(C2O4)3(s)":1.18848,
        "Y2(C2O4)3(s)":4.6740,
        "Sm2(C2O4)3(s)":2.296176,
        "Sc2(C2O4)3(s)":6.42030,
        "Pr2(C2O4)3(s)":2.09604,
        "Dy2(C2O4)3(s)":4.8608,
        "Ca(C2O4)(s)":14.49274,
        "FE2(C2O4)3(s)":8.659561}


for i in range(len(oxalate)):
    Lai = exp(-(eps["La2(C2O4)3(s)"]/(oxalate[i]))**n_DA["La2(C2O4)3(s)"])
    Ndi = exp(-(eps["Nd2(C2O4)3(s)"]/(oxalate[i]))**n_DA["Nd2(C2O4)3(s)"]) 
    Gdi = exp(-(eps["Gd2(C2O4)3(s)"]/(oxalate[i]))**n_DA["Gd2(C2O4)3(s)"]) 
    Cei = exp(-(eps["Ce2(C2O4)3(s)"]/(oxalate[i]))**n_DA["Ce2(C2O4)3(s)"]) 
    Yi = exp(-(eps["Y2(C2O4)3(s)"]/(oxalate[i]))**n_DA["Y2(C2O4)3(s)"]) 
    Smi = exp(-(eps["Sm2(C2O4)3(s)"]/(oxalate[i]))**n_DA["Sm2(C2O4)3(s)"]) 
    Sci = exp(-(eps["Sc2(C2O4)3(s)"]/(oxalate[i]))**n_DA["Sc2(C2O4)3(s)"]) 
    Pri = exp(-(eps["Pr2(C2O4)3(s)"]/(oxalate[i]))**n_DA["Pr2(C2O4)3(s)"]) 
    Dyi = exp(-(eps["Dy2(C2O4)3(s)"]/(oxalate[i]))**n_DA["Dy2(C2O4)3(s)"]) 
    Cai = exp(-(eps["Ca(C2O4)(s)"]/(oxalate[i]))**n_DA["Ca(C2O4)(s)"]) 
    Fei = exp(-(eps["FE2(C2O4)3(s)"]/(oxalate[i]))**n_DA["FE2(C2O4)3(s)"])
    
    La.append(Lai)
    Nd.append(Ndi)
    Gd.append(Gdi)
    Ce.append(Cei)
    Y.append(Yi)
    Sm.append(Smi)
    Sc.append(Sci)
    Pr.append(Pri)
    Dy.append(Dyi)
    Ca.append(Cai)
    Fe.append(Fei)

## 5.5 Make New Plot with the Model Values

In this section, model predictions are plotted together with
experimental data.

Good fit indicates that the estimated parameters provide a
reasonable model for simulating the precipitation behavior.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
ax.plot(data["Oxalate-2"],data["La2(C2O4)3(s)"],label="La2(C2O4)3(s)")
ax.plot(data["Oxalate-2"],data["Nd2(C2O4)3(s)"],label="Nd2(C2O4)3(s)")
ax.plot(data["Oxalate-2"],data["Gd2(C2O4)3(s)"],label="Gd2(C2O4)3(s)")
ax.plot(data["Oxalate-2"],data["Ce2(C2O4)3(s)"],label="Ce2(C2O4)3(s)")
ax.plot(data["Oxalate-2"],data["Y2(C2O4)3(s)"],label="Y2(C2O4)3(s)")
ax.plot(data["Oxalate-2"],data["Sm2(C2O4)3(s)"],label="Sm2(C2O4)3(s)")
ax.plot(data["Oxalate-2"],data["Sc2(C2O4)3(s)"],label="Sc2(C2O4)3(s)")
ax.plot(data["Oxalate-2"],data["Pr2(C2O4)3(s)"],label="Pr2(C2O4)3(s)")
ax.plot(data["Oxalate-2"],data["Dy2(C2O4)3(s)"],label="Dy2(C2O4)3(s)")
ax.plot(data["Oxalate-2"],data["Ca(C2O4)(s)"],label="Ca(C2O4)(s)")
ax.plot(data["Oxalate-2"],data["FE2(C2O4)3(s)"],label="FE2(C2O4)3(s)")
ax.scatter(oxalate,La)
ax.scatter(oxalate,Nd)
ax.scatter(oxalate,Gd)
ax.scatter(oxalate,Ce)
ax.scatter(oxalate,Y)
ax.scatter(oxalate,Sm)
ax.scatter(oxalate,Sc)
ax.scatter(oxalate,Pr)
ax.scatter(oxalate,Dy)
ax.scatter(oxalate,Ca)
ax.scatter(oxalate,Fe)

plt.xlabel('Acid Dosage')
plt.ylabel('Conversion')
plt.title('Conversion of Nd to ND oxalate in precipitator')

ax.legend(title='CM')
plt.show

## 5.6 Compare Model Predictions with Experimental Data 

The model predictions are compared with
experimental data reported in the report from University of Kentucky:

Steven Keim, "Production of salable rare earths products from coal and coal byproducts in the U.S.
using advanced separation processes", 2019




In [ ]:
oxalate1 = 6.4 

La1 = []
Nd1 = []
Gd1 = []
Ce1 = []
Y1 = []
Sm1 = []
Sc1 = []
Pr1 = []
Dy1 = []

La1 = exp(-(eps["La2(C2O4)3(s)"]/(oxalate1))**n_DA["La2(C2O4)3(s)"])
Nd1 = exp(-(eps["Nd2(C2O4)3(s)"]/(oxalate1))**n_DA["Nd2(C2O4)3(s)"]) 
Gd1 = exp(-(eps["Gd2(C2O4)3(s)"]/(oxalate1))**n_DA["Gd2(C2O4)3(s)"]) 
Ce1 = exp(-(eps["Ce2(C2O4)3(s)"]/(oxalate1))**n_DA["Ce2(C2O4)3(s)"]) 
Y1 = exp(-(eps["Y2(C2O4)3(s)"]/(oxalate1))**n_DA["Y2(C2O4)3(s)"]) 
Sm1 = exp(-(eps["Sm2(C2O4)3(s)"]/(oxalate1))**n_DA["Sm2(C2O4)3(s)"]) 
Sc1 = exp(-(eps["Sc2(C2O4)3(s)"]/(oxalate1))**n_DA["Sc2(C2O4)3(s)"]) 
Pr1 = exp(-(eps["Pr2(C2O4)3(s)"]/(oxalate1))**n_DA["Pr2(C2O4)3(s)"]) 
Dy1 = exp(-(eps["Dy2(C2O4)3(s)"]/(oxalate1))**n_DA["Dy2(C2O4)3(s)"]) 
Ca1 = exp(-(eps["Ca(C2O4)(s)"]/(oxalate1))**n_DA["Ca(C2O4)(s)"]) 
Fe1 = exp(-(eps["FE2(C2O4)3(s)"]/(oxalate1))**n_DA["FE2(C2O4)3(s)"])

bsarcm = np.array([La1,Nd1,Gd1,Ce1,Y1,Sm1,Sc1,Pr1,Dy1,Ca1,Fe1])

## 5.7 Make the Plot for Comparing Model Predictions and Experimental Results

In this section, the comparison plot is made to compare the model predictions based on
optimized parameters with experimental results from the UKy report.

The plot summarizes the performance of the multi-species parameter
estimation model and provides a validation for the model.

In [ ]:
critical_minerals = [
    "La2(C2O4)3(s)", 
    "Nd2(C2O4)3(s)", 
    "Gd2(C2O4)3(s)", 
    "Ce2(C2O4)3(s)", 
    "Y2(C2O4)3(s)", 
    "Sm2(C2O4)3(s)", 
    "Sc2(C2O4)3(s)", 
    "Pr2(C2O4)3(s)", 
    "Dy2(C2O4)3(s)",
    "Ca(C2O4)(s)",
    "FE2(C2O4)3(s)"]

# Including arrays with data from report
datacm = [0.94,0.99,0.96,0.97,0.85,0.95,0.4281,0.99,0.85,0.21,0.15]
datax = [6.4,6.4,6.4,6.4,6.4,6.4,6.4,6.4,6.4,6.4,6.4]

X_axis = np.arange(len(critical_minerals)) 

fig = plt.figure(figsize=[20,3])
ax = fig.add_subplot(1, 1, 1)

ax.bar(X_axis - 0.2,datacm, 0.4, label = 'report')
ax.bar(X_axis + 0.2,bsarcm, 0.4, label = 'model')

plt.xticks(X_axis, critical_minerals)
plt.xlabel("CM oxalate") 
plt.ylabel("% Recovery") 
plt.legend()
plt.show